[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/langchain-ai/langchain-academy/blob/main/module-0/basics.ipynb) [![Open in LangChain Academy](https://cdn.prod.website-files.com/65b8cd72835ceeacd4449a53/66e9eba12c7b7688aa3dbb5e_LCA-badge-green.svg)](https://academy.langchain.com/courses/take/intro-to-langgraph/lessons/56295530-getting-set-up-video-guide)

# LangChain Academy

Welcome to LangChain Academy! 

## Context

At LangChain, we aim to make it easy to build LLM applications. One type of LLM application you can build is an agent. There’s a lot of excitement around building agents because they can automate a wide range of tasks that were previously impossible. 

In practice though, it is incredibly difficult to build systems that reliably execute on these tasks. As we’ve worked with our users to put agents into production, we’ve learned that more control is often necessary. You might need an agent to always call a specific tool first or use different prompts based on its state. 

To tackle this problem, we’ve built [LangGraph](https://docs.langchain.com/oss/python/langgraph/overview) — a framework for building agent and multi-agent applications. Separate from the LangChain package, LangGraph’s core design philosophy is to help developers add better precision and control into agent workflows, suitable for the complexity of real-world systems.

## Course Structure

The course is structured as a set of modules, with each module focused on a particular theme related to LangGraph. You will see a folder for each module, which contains a series of notebooks. A video will accompany each notebook to help walk through the concepts, but the notebooks are also stand-alone, meaning that they contain explanations and can be viewed independently of the videos. Each module folder also contains a `studio` folder, which contains a set of graphs that can be loaded into [LangSmith Studio](https://docs.langchain.com/langsmith/quick-start-studio), our IDE for building LangGraph applications.

## Setup

Before you begin, please follow the instructions in the `README` to create an environment and install dependencies.

## Chat models

In this course, we will use chat models, which take a sequence of messages as input and return messages as output. LangChain supports many models through third-party integrations.

For this local setup, we will use Ollama with locally installed models. This avoids any requirement for an OpenAI API key and allows the notebooks to run against models available on your machine.

The examples below use:

- `qwen3:14b` as the primary chat model
- `qwen2.5-coder:7b` as the lighter local chat model

Let's install the required LangChain packages for Ollama and the course examples.

In [69]:
%%capture --no-stderr
%pip install --quiet -U langchain-ollama langchain_core langchain_community langchain-tavily

In [70]:
import os
import getpass
from dotenv import load_dotenv

load_dotenv()  # loads .env if present

def _set_env(var: str):
    if not os.environ.get(var):
        os.environ[var] = getpass.getpass(f"{var}: ")

[Here](https://docs.langchain.com/oss/python/langchain/models) is a useful how-to for all the things that you can do with chat models, but we'll show a few highlights below. If you've run `pip install -r requirements.txt` as noted in the README, then you've installed the `langchain-openai` package. With this, we can instantiate our `ChatOpenAI` model object. You can see pricing for various models [here](https://openai.com/api/pricing/). The notebooks will default to `gpt-4o` because it offers a good balance of quality, price, and speed, but you can also opt for the lower-priced `gpt-3.5` series or more recent models.

There are [a few standard parameters](https://docs.langchain.com/oss/python/langchain/models#parameters) that we can set with chat models. Two of the most common are:

* `model`: the name of the model
* `temperature`: the sampling temperature

`Temperature` controls the randomness or creativity of the model's output where low temperature (close to 0) is more deterministic and focused outputs. This is good for tasks requiring accuracy or factual responses. High temperature (close to 1) is good for creative tasks or generating varied responses. 

In [71]:
from langchain_ollama import ChatOllama

qwen3_chat = ChatOllama(
    model="qwen3:14b",
    base_url="http://localhost:11434",
    temperature=0,
)

qwen_coder_chat = ChatOllama(
    model="qwen2.5-coder:7b",
    base_url="http://localhost:11434",
    temperature=0,
)

Chat models in LangChain have a number of [default methods](https://reference.langchain.com/python/langchain_core/runnables). For the most part, we'll be using:

* [stream](https://docs.langchain.com/oss/python/langchain/models#stream): stream back chunks of the response
* [invoke](https://docs.langchain.com/oss/python/langchain/models#invoke): call the chain on an input

And, as mentioned, chat models take [messages](https://docs.langchain.com/oss/python/langchain/messages) as input. Messages have a role (that describes who is saying the message) and a content property. We'll be talking a lot more about this later, but here let's just show the basics.

In [72]:
from langchain_core.messages import HumanMessage

# Create a message
msg = HumanMessage(content="Hello world", name="Lance")

# Message list
messages = [msg]

# Invoke the model with a list of messages 
qwen3_chat.invoke(messages)

AIMessage(content='Hello! 😊 How can I assist you today? Let me know if you have any questions or need help with something!', additional_kwargs={}, response_metadata={'model': 'qwen3:14b', 'created_at': '2026-05-29T11:36:04.5714261Z', 'done': True, 'done_reason': 'stop', 'total_duration': 19241072600, 'load_duration': 8201622800, 'prompt_eval_count': 12, 'prompt_eval_duration': 379703500, 'eval_count': 123, 'eval_duration': 10607249500, 'logprobs': None, 'model_name': 'qwen3:14b', 'model_provider': 'ollama'}, id='lc_run--019e7384-d3ad-71a1-9609-e0a84e115433-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 12, 'output_tokens': 123, 'total_tokens': 135})

We get an `AIMessage` response. Also, note that we can just invoke a chat model with a string. When a string is passed in as input, it is converted to a `HumanMessage` and then passed to the underlying model.


In [73]:
qwen3_chat.invoke("hello world")

AIMessage(content='Hello! 😊 It looks like you\'re starting with "Hello, World!" — a classic first step in programming! Are you working on your first program, or is there something specific you\'d like help with? I\'m here to assist with code, explanations, or any other questions you might have! 🚀', additional_kwargs={}, response_metadata={'model': 'qwen3:14b', 'created_at': '2026-05-29T11:36:31.0777479Z', 'done': True, 'done_reason': 'stop', 'total_duration': 26485163200, 'load_duration': 4787448900, 'prompt_eval_count': 12, 'prompt_eval_duration': 386109900, 'eval_count': 236, 'eval_duration': 21235625300, 'logprobs': None, 'model_name': 'qwen3:14b', 'model_provider': 'ollama'}, id='lc_run--019e7385-1eef-74f3-893e-83a628a86489-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 12, 'output_tokens': 236, 'total_tokens': 248})

In [74]:
qwen_coder_chat.invoke("hello world")

AIMessage(content='Hello! How can I assist you today?', additional_kwargs={}, response_metadata={'model': 'qwen2.5-coder:7b', 'created_at': '2026-05-29T11:36:37.4625885Z', 'done': True, 'done_reason': 'stop', 'total_duration': 6364498200, 'load_duration': 6113811400, 'prompt_eval_count': 31, 'prompt_eval_duration': 62700900, 'eval_count': 10, 'eval_duration': 171778300, 'logprobs': None, 'model_name': 'qwen2.5-coder:7b', 'model_provider': 'ollama'}, id='lc_run--019e7385-8678-7b52-aed4-36c4ae28f909-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 31, 'output_tokens': 10, 'total_tokens': 41})

The interface is consistent across all chat models and models are typically initialized once at the start up each notebooks. 

So, you can easily switch between models without changing the downstream code if you have strong preference for another provider.


## Search Tools

You'll also see [Tavily](https://tavily.com/) in the README, which is a search engine optimized for LLMs and RAG, aimed at efficient, quick, and persistent search results. As mentioned, it's easy to sign up and offers a generous free tier. Some lessons (in Module 4) will use Tavily by default but, of course, other search tools can be used if you want to modify the code for yourself.

In [75]:
_set_env("TAVILY_API_KEY")

In [76]:
from langchain_tavily import TavilySearch

tavily_search = TavilySearch(
    max_results=3,
    include_answer=True,
    include_raw_content=True,
)

data = tavily_search.invoke({"query": "What is LangGraph?"})
search_docs = data.get("results", data)

In [77]:
search_docs

[{'url': 'https://www.ibm.com/think/topics/langgraph',
  'title': 'What is LangGraph? - IBM',
  'content': 'LangGraph, created by LangChain, is an open source AI agent framework designed to build, deploy and manage complex generative AI agent workflows. It provides a set of tools and libraries that enable users to create, run and optimize large language models (LLMs) in a scalable and efficient manner. At its core, LangGraph uses the power of graph-based architectures to model and manage the intricate relationships between various components of an AI agent workflow. The following example can offer a clearer understanding of LangGraph: Think about these graph-based architectures as a powerful configurable map, a “Super-Map.” Users can envision the AI workflow as being “The Navigator” of this “Super-Map.” Finally, in this example, the user is “The Cartographer.” In this sense, the navigator charts out the optimal routes between points on the “Super-Map,” all of which are created by “The 